# Evaluación Final — Módulo 03

Dataset **`housing.csv`** (censo de California, 1990): información de viviendas por distrito. Variable target: `median_house_value`.

Consignas completas en [evaluacion_final_modulo.md](evaluacion_final_modulo.md).

**Preprocesamiento requerido antes de resolver el cuestionario:**
1. Rellenar los valores faltantes con la **media** de la variable.
2. **Estandarizar** las variables numéricas con `StandardScaler()` (dropeando la categórica).
3. Codificar la variable categórica con `OneHotEncoder()`.
4. Unificar todo en un **único dataframe distinto del original**.

**Split:** 20% para test, `random_state=42`.

> Nota: siguiendo la consigna literal, se estandarizan **todas las variables numéricas, incluido el target** `median_house_value`. El **R²** no se ve afectado (es invariante a la escala de `y`), pero el intercepto y el MSE/RMSE quedan en unidades estandarizadas.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv('../datasets/housing.csv')
df.head()

In [ ]:
# Valores faltantes por columna
df.isnull().sum()

## 1. Rellenar los valores faltantes con la media

Solo `total_bedrooms` tiene nulos (207).

In [ ]:
df['total_bedrooms'] = df['total_bedrooms'].fillna(df['total_bedrooms'].mean())
df.isnull().sum().sum()

## 2-4. Estandarizar numéricas, codificar categórica y unificar

- `StandardScaler` sobre **todas las variables numéricas** (incluido el target `median_house_value`), dropeando la categórica.
- `OneHotEncoder` sobre `ocean_proximity`.
- Concatenamos todo en un **único dataframe nuevo** (`df_prep`).

In [ ]:
target = 'median_house_value'

# Variables numericas: todas menos la categorica ocean_proximity (incluye el target)
num_cols = df.drop(columns='ocean_proximity').columns.tolist()
num_cols

In [ ]:
# 2. Estandarizacion de las variables numericas (incluye el target, como pide la consigna)
scaler = StandardScaler()
num_scaled = pd.DataFrame(
    scaler.fit_transform(df[num_cols]),
    columns=num_cols,
    index=df.index
)
num_scaled.head()

In [ ]:
# 3. Codificacion de la variable categorica
enc = OneHotEncoder(sparse_output=False)
cat_encoded = pd.DataFrame(
    enc.fit_transform(df[['ocean_proximity']]),
    columns=enc.get_feature_names_out(['ocean_proximity']),
    index=df.index
)
cat_encoded.head()

In [ ]:
# 4. Dataframe unificado (distinto del original): numericas escaladas + categorica codificada
df_prep = pd.concat([num_scaled, cat_encoded], axis=1)
print(df_prep.shape)
df_prep.head()

## Train / Test split

20% para test, `random_state=42`.

In [ ]:
X = df_prep.drop(columns=target)
y = df_prep[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

## Cuestionario

A partir de acá se resuelven las consignas del cuestionario (creación y evaluación de modelos, etc.) usando `X_train`, `X_test`, `y_train`, `y_test`. Se irán agregando a medida que se pidan.

### Pregunta 1 — Árbol de decisión

Crear un árbol de decisión de regresión con **`max_depth=5`**, **`min_samples_split=20`** y **`random_state=20`**. Entrenar, evaluar y reportar el **R²** sobre test.

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score

tree = DecisionTreeRegressor(max_depth=5, min_samples_split=20, random_state=20)
tree.fit(X_train, y_train)
preds = tree.predict(X_test)

r2 = r2_score(y_test, preds)
print("R2:", r2)

### Pregunta 2 — Duplicar la profundidad

Si se duplica la profundidad del árbol del punto anterior (`max_depth=10`), ¿el modelo **empeora**, **no cambia** o **mejora**?

In [ ]:
# Duplicamos la profundidad: max_depth=10 (mismos min_samples_split y random_state)
tree10 = DecisionTreeRegressor(max_depth=10, min_samples_split=20, random_state=20)
tree10.fit(X_train, y_train)
preds10 = tree10.predict(X_test)

r2_10 = r2_score(y_test, preds10)
print("R2 (max_depth=5): ", round(r2, 4))
print("R2 (max_depth=10):", round(r2_10, 4))

### Pregunta 3 — Regresión lineal (por defecto)

Crear una `LinearRegression()` con valores por defecto para aproximar `median_house_value`. Entrenar, evaluar y reportar el **R²**.

In [ ]:
from sklearn.linear_model import LinearRegression

linreg = LinearRegression()
linreg.fit(X_train, y_train)
preds_lr = linreg.predict(X_test)

r2_lr = r2_score(y_test, preds_lr)
print("R2:", round(r2_lr, 4))

### Pregunta 4 — Término independiente

¿A cuánto se aproxima el **término independiente** (intercepto) del regresor lineal del punto anterior?

In [ ]:
# Termino independiente (intercepto) del regresor lineal del punto anterior
print("Intercepto:", round(linreg.intercept_, 4))